In [1]:
# ============================================================
# DATA JOBS MARKET ANALYSIS
# ============================================================
# Business Goal:
# Analyze job-market demand, salaries, remote work,
# geographic trends, salary benchmarks, and job opportunities.
# ============================================================


# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
from datasets import load_dataset


# ============================================================
# 2. LOAD DATA
# ============================================================

dataset = load_dataset('lukebarousse/data_jobs')

df = dataset['train'].to_pandas()



README.md:   0%|          | 0.00/3.25k [00:00<?, ?B/s]

data_jobs.csv: reconstructing file:   0%|          |  0.00B /  231MB            

data_jobs.csv: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/785741 [00:00<?, ? examples/s]

In [2]:
# ============================================================
# 3. DATA CLEANING
# ============================================================

# Convert job posting date from string/object to datetime
df['job_posted_date'] = pd.to_datetime(df['job_posted_date'])

print("Dataset loaded successfully!")
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")


Dataset loaded successfully!
Rows: 785741
Columns: 17


In [3]:
# ============================================================
# 4. DATA QUALITY AUDIT
# ============================================================

print("\n--- DATASET SHAPE ---")
print(df.shape)

print("\n--- DATA TYPES ---")
print(df.dtypes)

print("\n--- MISSING VALUES ---")
print(df.isna().sum())

print("\n--- UNIQUE VALUES ---")
print(df.nunique())

print("\n--- DUPLICATE ROWS ---")
print(df.duplicated().sum())



--- DATASET SHAPE ---
(785741, 17)

--- DATA TYPES ---
job_title_short                  object
job_title                        object
job_location                     object
job_via                          object
job_schedule_type                object
job_work_from_home                 bool
search_location                  object
job_posted_date          datetime64[ns]
job_no_degree_mention              bool
job_health_insurance               bool
job_country                      object
salary_rate                      object
salary_year_avg                 float64
salary_hour_avg                 float64
company_name                     object
job_skills                       object
job_type_skills                  object
dtype: object

--- MISSING VALUES ---
job_title_short               0
job_title                     1
job_location               1045
job_via                       8
job_schedule_type         12667
job_work_from_home            0
search_location               0
jo

In [4]:
# ============================================================
# 5. SALARY DATA AVAILABILITY
# ============================================================
# Business Question:
# Which job titles have the best salary-data availability?


salary_availability = df.groupby('job_title_short').agg(
    jobs_with_salary=('salary_year_avg', 'count'),
    total_jobs=('job_title_short', 'count')
)

salary_availability['salary_data_pct'] = (
    salary_availability['jobs_with_salary'] /
    salary_availability['total_jobs'] * 100
)

salary_availability = salary_availability.sort_values(
    'salary_data_pct',
    ascending=False
)

print("\n--- SALARY DATA AVAILABILITY ---")
print(salary_availability)




--- SALARY DATA AVAILABILITY ---
                           jobs_with_salary  total_jobs  salary_data_pct
job_title_short                                                         
Senior Data Scientist                  1690       36957         4.572882
Machine Learning Engineer               576       14080         4.090909
Senior Data Analyst                    1131       29216         3.871166
Senior Data Engineer                   1591       44563         3.570226
Data Scientist                         5922      172286         3.437308
Data Analyst                           5451      196075         2.780059
Data Engineer                          4500      186241         2.416224
Business Analyst                        610       49063         1.243299
Software Engineer                       467       44929         1.039418
Cloud Engineer                           65       12331         0.527127


In [5]:
# ============================================================
# 6. JOB MARKET DEMAND
# ============================================================
# Business Question:
# Which job titles have the highest demand?


job_demand = df.groupby('job_title_short').agg(
    total_jobs=('job_title_short', 'count')
).sort_values(
    'total_jobs',
    ascending=False
)

print("\n--- TOP JOB TITLES BY DEMAND ---")
print(job_demand.head(10))



--- TOP JOB TITLES BY DEMAND ---
                           total_jobs
job_title_short                      
Data Analyst                   196075
Data Engineer                  186241
Data Scientist                 172286
Business Analyst                49063
Software Engineer               44929
Senior Data Engineer            44563
Senior Data Scientist           36957
Senior Data Analyst             29216
Machine Learning Engineer       14080
Cloud Engineer                  12331


In [6]:
# ============================================================
# 7. GEOGRAPHIC JOB & SALARY ANALYSIS
# ============================================================
# Business Question:
# Which countries have a large job market and high salaries?


country_analysis = df.groupby('job_country').agg(
    total_jobs=('job_country', 'count'),
    avg_salary=('salary_year_avg', 'mean'),
    median_salary=('salary_year_avg', 'median')
)

country_analysis = country_analysis[
    country_analysis['total_jobs'] >= 500
].sort_values(
    'median_salary',
    ascending=False
)

print("\n--- COUNTRIES WITH LARGE JOB MARKETS ---")
print(country_analysis)



--- COUNTRIES WITH LARGE JOB MARKETS ---
              total_jobs     avg_salary  median_salary
job_country                                           
Belarus              543  400000.000000      400000.00
Russia              3743  292500.000000      300000.00
Tunisia              705  118754.600000      147500.00
Chile               8118  124918.454545      146000.00
Sudan              21781  134051.577942      129500.00
...                  ...            ...            ...
China               2534   93897.625000       52295.25
Ecuador              656            NaN            NaN
Kuwait               698            NaN            NaN
Qatar               1406            NaN            NaN
Saudi Arabia        3401            NaN            NaN

[75 rows x 3 columns]


In [7]:
# ============================================================
# 8. MONTHLY JOB DEMAND
# ============================================================
# Business Question:
# How does job-posting volume change throughout the year?


df['month'] = df['job_posted_date'].dt.month

monthly_demand = df.groupby('month').agg(
    total_jobs=('job_posted_date', 'count')
).sort_index()

print("\n--- MONTHLY JOB DEMAND ---")
print(monthly_demand)




--- MONTHLY JOB DEMAND ---
       total_jobs
month            
1           91822
2           64578
3           64084
4           62919
5           52104
6           61572
7           63777
8           75162
9           62359
10          66611
11          64450
12          56303


In [8]:
# ============================================================
# 9. REMOTE VS NON-REMOTE ANALYSIS
# ============================================================
# Business Question:
# How do remote and non-remote jobs compare?


df['wfh_category'] = df['job_work_from_home'].map({
    True: 'Remote',
    False: 'Non-Remote'
})

wfh_analysis = df.groupby('wfh_category').agg(
    total_jobs=('wfh_category', 'count'),
    jobs_with_salary=('salary_year_avg', 'count'),
    avg_salary=('salary_year_avg', 'mean'),
    median_salary=('salary_year_avg', 'median')
)

print("\n--- REMOTE VS NON-REMOTE ---")
print(wfh_analysis)




--- REMOTE VS NON-REMOTE ---
              total_jobs  jobs_with_salary     avg_salary  median_salary
wfh_category                                                            
Non-Remote        716189             18724  121830.018124     115000.000
Remote             69552              3279  131601.899679     128829.625


In [9]:
# ============================================================
# 10. COUNTRY + JOB TITLE ANALYSIS
# ============================================================
# Business Question:
# Which country/job-title combinations offer strong
# demand and high salaries?


country_title_analysis = df.groupby(
    ['job_country', 'job_title_short']
).agg(
    total_jobs=('job_title_short', 'count'),
    avg_salary=('salary_year_avg', 'mean'),
    median_salary=('salary_year_avg', 'median')
)

country_title_analysis = country_title_analysis[
    (country_title_analysis['total_jobs'] >= 20) &
    (country_title_analysis['avg_salary'] >= 100000)
]

country_title_analysis = country_title_analysis.sort_values(
    'median_salary',
    ascending=False
)

print("\n--- TOP COUNTRY + JOB TITLE COMBINATIONS ---")
print(country_title_analysis.head(15))



--- TOP COUNTRY + JOB TITLE COMBINATIONS ---
                                                total_jobs     avg_salary  \
job_country          job_title_short                                        
Belarus              Data Analyst                      126  400000.000000   
Puerto Rico          Senior Data Scientist              51  375000.000000   
Russia               Software Engineer                 348  320000.000000   
                     Data Scientist                    630  285000.000000   
                     Cloud Engineer                     32  280000.000000   
Argentina            Cloud Engineer                    322  197500.000000   
Cyprus               Data Scientist                     64  191000.000000   
Colombia             Machine Learning Engineer         221  201666.666667   
                     Cloud Engineer                    298  182500.000000   
Sri Lanka            Machine Learning Engineer          35  182325.000000   
Argentina            Software 

In [10]:
# ============================================================
# 11. HIGH-DEMAND + HIGH-SALARY JOBS
# ============================================================
# Business Question:
# Which job titles combine strong demand with high salaries?


high_value_jobs = df.groupby('job_title_short').agg(
    total_jobs=('job_title_short', 'count'),
    avg_salary=('salary_year_avg', 'mean'),
    median_salary=('salary_year_avg', 'median')
)

high_value_jobs = high_value_jobs[
    (high_value_jobs['total_jobs'] >= 50) &
    (high_value_jobs['avg_salary'] >= 90000)
]

high_value_jobs = high_value_jobs.sort_values(
    'median_salary',
    ascending=False
)

print("\n--- HIGH-DEMAND + HIGH-SALARY JOBS ---")
print(high_value_jobs)



--- HIGH-DEMAND + HIGH-SALARY JOBS ---
                           total_jobs     avg_salary  median_salary
job_title_short                                                    
Senior Data Scientist           36957  154206.292996       155500.0
Senior Data Engineer            44563  145840.611624       147500.0
Data Scientist                 172286  135988.837171       127500.0
Data Engineer                  186241  130125.604250       125000.0
Senior Data Analyst             29216  113911.363665       111175.0
Machine Learning Engineer       14080  126774.315972       106415.0
Software Engineer               44929  113393.760054        99150.0
Cloud Engineer                  12331  111268.453846        90000.0
Data Analyst                   196075   93841.907854        90000.0
Business Analyst                49063   91082.612833        85000.0


In [11]:
# ============================================================
# 12. SALARY DISTRIBUTION & OUTLIERS
# ============================================================
# Business Question:
# What does the salary distribution look like,
# and are there unusually high salaries?


salary = df['salary_year_avg'].dropna()

min_salary = salary.min()
q1 = salary.quantile(0.25)
median_salary = salary.median()
q3 = salary.quantile(0.75)
max_salary = salary.max()

iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

outliers = salary[salary > upper_bound]

outlier_count = outliers.count()

outlier_percentage = (
    outlier_count / salary.count() * 100
)

salary_summary = pd.DataFrame({
    'min_salary': [min_salary],
    'Q1': [q1],
    'median_salary': [median_salary],
    'Q3': [q3],
    'max_salary': [max_salary],
    'IQR': [iqr],
    'lower_bound': [lower_bound],
    'upper_bound': [upper_bound],
    'outlier_count': [outlier_count],
    'outlier_percentage': [outlier_percentage]
})

print("\n--- SALARY DISTRIBUTION ---")
print(salary_summary)




--- SALARY DISTRIBUTION ---
   min_salary       Q1  median_salary        Q3  max_salary      IQR  \
0     15000.0  90000.0       115000.0  150000.0    960000.0  60000.0   

   lower_bound  upper_bound  outlier_count  outlier_percentage  
0          0.0     240000.0            416            1.890651  


In [12]:
# ============================================================
# 13. SALARY BENCHMARKING BY COUNTRY
# ============================================================
# Business Question:
# Is an individual job's salary above or below
# the median salary in its country?


df['country_median_salary'] = (
    df.groupby('job_country')['salary_year_avg']
      .transform('median')
)

df['salary_vs_country_median'] = (
    df['salary_year_avg'] -
    df['country_median_salary']
)

salary_benchmark = df[
    [
        'job_country',
        'job_title_short',
        'salary_year_avg',
        'country_median_salary',
        'salary_vs_country_median'
    ]
]

print("\n--- SALARY BENCHMARKING ---")
print(salary_benchmark.head())




--- SALARY BENCHMARKING ---
     job_country       job_title_short  salary_year_avg  \
0  United States  Senior Data Engineer              NaN   
1         Mexico          Data Analyst              NaN   
2        Germany         Data Engineer              NaN   
3  United States         Data Engineer              NaN   
4          Sudan         Data Engineer              NaN   

   country_median_salary  salary_vs_country_median  
0               119187.5                       NaN  
1               109500.0                       NaN  
2               111175.0                       NaN  
3               119187.5                       NaN  
4               129500.0                       NaN  


In [13]:

# ============================================================
# 14. JOB MARKET TRENDS
# ============================================================
# Business Question:
# How do job demand and salaries change by month?


monthly_trends = df.groupby('month').agg(
    total_jobs=('job_posted_date', 'count'),
    avg_salary=('salary_year_avg', 'mean'),
    median_salary=('salary_year_avg', 'median')
).sort_index()

print("\n--- MONTHLY JOB MARKET TRENDS ---")
print(monthly_trends)



--- MONTHLY JOB MARKET TRENDS ---
       total_jobs     avg_salary  median_salary
month                                          
1           91822  123007.422072       115000.0
2           64578  122913.927237       115000.0
3           64084  122947.810047       117050.0
4           62919  121856.551050       115000.0
5           52104  123656.747729       115000.0
6           61572  124321.135360       115000.0
7           63777  123470.572994       115000.0
8           75162  125819.429100       115000.0
9           62359  125222.984594       117750.0
10          66611  124606.653605       117803.0
11          64450  119817.184014       112392.0
12          56303  120199.423414       112500.0


In [14]:
# ============================================================
# 15. MONTHLY TRENDS BY JOB TITLE
# ============================================================
# Business Question:
# Which job titles have strong demand and salaries
# in different months?


monthly_title_analysis = df.groupby(
    ['month', 'job_title_short']
).agg(
    total_jobs=('job_title_short', 'count'),
    avg_salary=('salary_year_avg', 'mean'),
    median_salary=('salary_year_avg', 'median')
)

monthly_title_analysis = monthly_title_analysis[
    (monthly_title_analysis['total_jobs'] >= 20) &
    (monthly_title_analysis['avg_salary'] >= 90000)
]

monthly_title_analysis = monthly_title_analysis.sort_values(
    ['month', 'median_salary'],
    ascending=[True, False]
)

print("\n--- MONTHLY JOB TITLE ANALYSIS ---")
print(monthly_title_analysis.head(15))



--- MONTHLY JOB TITLE ANALYSIS ---
                                 total_jobs     avg_salary  median_salary
month job_title_short                                                    
1     Senior Data Scientist            4644  155702.728365       157500.0
      Senior Data Engineer             5041  149084.859256       147500.0
      Data Scientist                  20760  136017.017966       130000.0
      Data Engineer                   21419  129349.091732       125000.0
      Senior Data Analyst              3696  113017.852732       111175.0
      Machine Learning Engineer        1386  122632.192857       104668.0
      Cloud Engineer                   1295  119575.000000        99550.0
      Data Analyst                    23585   92075.148339        89565.0
      Business Analyst                 4874   94747.064516        87500.0
      Software Engineer                5122   99408.350000        84150.0
2     Senior Data Scientist            2994  147643.119205       152000.0
  

In [15]:
# ============================================================
# 16. REMOTE SALARY ADVANTAGE
# ============================================================
# Business Question:
# Which job titles pay more for remote positions?


remote = df[
    df['wfh_category'] == 'Remote'
].groupby('job_title_short').agg(
    total_jobs=('job_title_short', 'count'),
    avg_salary=('salary_year_avg', 'mean')
)

non_remote = df[
    df['wfh_category'] == 'Non-Remote'
].groupby('job_title_short').agg(
    total_jobs=('job_title_short', 'count'),
    avg_salary=('salary_year_avg', 'mean')
)

# Keep job titles with enough observations
remote = remote[
    remote['total_jobs'] >= 30
]

non_remote = non_remote[
    non_remote['total_jobs'] >= 30
]

# Pandas aligns the job-title indexes automatically
remote_salary = remote['avg_salary'].rename(
    'remote_avg_salary'
)

non_remote_salary = non_remote['avg_salary'].rename(
    'non_remote_avg_salary'
)

salary_advantage = (
    remote_salary -
    non_remote_salary
).rename('salary_advantage')

remote_salary_analysis = pd.concat(
    [
        remote_salary,
        non_remote_salary,
        salary_advantage
    ],
    axis=1
)

remote_salary_analysis = remote_salary_analysis.sort_values(
    'salary_advantage',
    ascending=False
)

print("\n--- REMOTE SALARY ADVANTAGE ---")
print(remote_salary_analysis.head(10))




--- REMOTE SALARY ADVANTAGE ---
                           remote_avg_salary  non_remote_avg_salary  \
job_title_short                                                       
Cloud Engineer                 147111.111111          105508.026786   
Machine Learning Engineer      148357.950000          125163.597015   
Senior Data Scientist          163968.789318          151774.688222   
Software Engineer              123006.793651          111894.697884   
Data Scientist                 144283.632179          134405.937420   
Business Analyst                97348.428571           90177.419940   
Senior Data Engineer           148200.284452          145330.070790   
Data Engineer                  132013.016769          129747.517613   
Data Analyst                    94531.957057           93754.956809   
Senior Data Analyst            113059.095068          114051.799273   

                           salary_advantage  
job_title_short                              
Cloud Engineer        

In [16]:
# ============================================================
# 17. TOP 3 HIGHEST-PAYING JOB TITLES BY COUNTRY
# ============================================================
# Business Question:
# What are the top 3 highest-paying job titles
# within each country?


country_title_rank = df.groupby(
    ['job_country', 'job_title_short']
).agg(
    total_jobs=('job_title_short', 'count'),
    avg_salary=('salary_year_avg', 'mean'),
    median_salary=('salary_year_avg', 'median')
)

# Remove combinations with too few jobs
country_title_rank = country_title_rank[
    country_title_rank['total_jobs'] >= 20
]

# Rank titles within each country
country_title_rank['salary_rank'] = (
    country_title_rank
    .groupby('job_country')['median_salary']
    .rank(
        method='dense',
        ascending=False
    )
)

# Keep top 3
top_3_by_country = country_title_rank[
    country_title_rank['salary_rank'] <= 3
]

top_3_by_country = top_3_by_country.sort_values(
    ['job_country', 'salary_rank']
)

print("\n--- TOP 3 JOB TITLES BY COUNTRY ---")
print(top_3_by_country)




--- TOP 3 JOB TITLES BY COUNTRY ---
                                  total_jobs   avg_salary  median_salary  \
job_country job_title_short                                                
Albania     Data Analyst                  36   49950.0000        49950.0   
Algeria     Data Engineer                 21   45000.0000        45000.0   
            Data Analyst                  22   44100.0000        44100.0   
Argentina   Cloud Engineer               322  197500.0000       197500.0   
            Software Engineer           1267  174500.0000       174500.0   
...                                      ...          ...            ...   
Uzbekistan  Data Scientist                50   30750.0000        30750.0   
Vietnam     Senior Data Engineer         220  127125.0000       147500.0   
            Senior Data Analyst           71  100512.5000       105837.5   
            Data Engineer                768   98508.1875        96773.0   
Zimbabwe    Data Analyst                  34   6300

In [17]:
# ============================================================
# 18. BEST OVERALL JOB OPPORTUNITIES
# ============================================================
# Business Question:
# Which jobs combine demand, salary, and remote opportunities?


job_opportunities = df.groupby(
    'job_title_short'
).agg(
    total_jobs=('job_title_short', 'count'),
    avg_salary=('salary_year_avg', 'mean'),
    median_salary=('salary_year_avg', 'median'),
    remote_jobs=('job_work_from_home', 'sum')
)

# Calculate remote percentage
job_opportunities['remote_pct'] = (
    job_opportunities['remote_jobs'] /
    job_opportunities['total_jobs'] * 100
)

# Apply business requirements
job_opportunities = job_opportunities[
    (job_opportunities['total_jobs'] >= 50) &
    (job_opportunities['avg_salary'] >= 90000) &
    (job_opportunities['remote_jobs'] >= 30)
]

# Rank highest-paying opportunities first
job_opportunities = job_opportunities.sort_values(
    'median_salary',
    ascending=False
)

print("\n--- BEST OVERALL JOB OPPORTUNITIES ---")
print(job_opportunities)


--- BEST OVERALL JOB OPPORTUNITIES ---
                           total_jobs     avg_salary  median_salary  \
job_title_short                                                       
Senior Data Scientist           36957  154206.292996       155500.0   
Senior Data Engineer            44563  145840.611624       147500.0   
Data Scientist                 172286  135988.837171       127500.0   
Data Engineer                  186241  130125.604250       125000.0   
Senior Data Analyst             29216  113911.363665       111175.0   
Machine Learning Engineer       14080  126774.315972       106415.0   
Software Engineer               44929  113393.760054        99150.0   
Cloud Engineer                  12331  111268.453846        90000.0   
Data Analyst                   196075   93841.907854        90000.0   
Business Analyst                49063   91082.612833        85000.0   

                           remote_jobs  remote_pct  
job_title_short                                     
S